# Evaluation basics

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

This notebook is a software tutorial using bundled example bars or a small deterministic fixture. It is not a trading recommendation or a research performance claim.

A return percentage alone hides path risk and trade quality. Here we calculate framework metrics from a small deterministic equity series and a compact trade ledger.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from evaluation.metrics.performance import compute_metrics

equity = np.array([10_000, 10_120, 10_060, 10_240, 10_190, 10_360], dtype=float)
trade_pnls = [120.0, -60.0, 180.0, -50.0, 170.0]
metrics = compute_metrics(
    equity, initial_balance=equity[0], trade_pnls=trade_pnls, periods_per_year=5
)
print(
    {
        "return_pct": round(metrics.total_return_pct, 2),
        "max_drawdown": round(metrics.max_drawdown, 4),
        "profit_factor": round(metrics.profit_factor, 2),
        "win_rate": round(metrics.win_rate, 2),
        "trades": metrics.n_trades,
    }
)

Figure: the equity curve above and fractional drawdown below, both indexed by evaluation step.

In [ ]:
running_peak = np.maximum.accumulate(equity)
drawdown = equity / running_peak - 1.0
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax1.plot(equity, marker="o", color="#2369a8")
ax1.set(title="Equity path", ylabel="Account currency")
ax2.fill_between(range(len(drawdown)), drawdown, 0, color="#b84a4a", alpha=0.35)
ax2.set(title="Drawdown from prior peak", ylabel="Fraction", xlabel="Evaluation step")
plt.tight_layout()
plt.show()

## Recompute path risk from equity

Metrics should be traceable to the underlying series. Here drawdown is equity divided by the running peak minus one, so it is non-positive and recovers only when a new high is reached.


In [ ]:
running_peak = np.maximum.accumulate(equity)
drawdown = equity / running_peak - 1.0
print({"first_equity": equity[0], "final_equity": equity[-1], "worst_drawdown": drawdown.min()})
assert drawdown[0] == 0.0 and (drawdown <= 0.0).all()

## Match trade outcomes to the curve

A path metric and a trade metric answer different questions. The curve captures the sequence and depth of losses; the trade list supports hit rate and profit factor. Do not infer a risk profile from final return alone.


In [ ]:
gross_profit = sum(x for x in trade_pnls if x > 0)
gross_loss = -sum(x for x in trade_pnls if x < 0)
print(
    {
        "trades": len(trade_pnls),
        "gross_profit": gross_profit,
        "gross_loss": gross_loss,
        "profit_factor": gross_profit / gross_loss,
    }
)
assert len(trade_pnls) == metrics.n_trades

## Interpret the output

This path has gains and losses, and the sequence affects drawdown even when the final return is positive. A full evaluation also inspects fills, breach events, open-position handling, and data coverage.

See [`compute_metrics`](../../api/domains/workflows), [evaluation methodology](../../research/evaluation_methodology), and [metrics reference](../../reference/metrics).